# BERT による固有表現認識（Token Classification）

書籍 第3章「系列ラベリング問題の解き方」（3-1-3 深層学習ベース、評価・エラー分析）に対応する演習ノートブックです。
[`ch3_morphology_ner.ipynb`](ch3_morphology_ner.ipynb)（形態素解析・固有表現認識の基礎）および [`ch3_ner_gazetteer_subword.ipynb`](ch3_ner_gazetteer_subword.ipynb)（ガゼッティアとサブワード・BIO ラベル整合）を発展させ、**日本語 BERT（`tohoku-nlp/bert-base-japanese-v3`）** を Token Classification タスクとしてファインチューニングします。

このノートブックでは以下を一貫して体験します。
1. **BIO 形式の日本語 NER データセット構築**（`PER` 人名・`ORG` 組織名・`LOC` 地名、計 114 文を層化分割）
2. **サブワード分割と BIO ラベルの整合（`-100` マスキング）**
3. **`AutoModelForTokenClassification` + `Trainer` によるファインチューニング**
4. **`seqeval` によるエンティティ単位 F1 評価とトークン単位正解率の比較**
5. **エラー分析と任意の日本語文に対する固有表現抽出**

> **実行環境の目安**: Google Colab の無料枠 GPU（T4）で約 1〜2 分で完了します。メニューの **「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」** を選択してから実行してください。

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU が利用可能です: {gpu_name}")
else:
    print(
        "⚠️ GPU が検出されませんでした（CPU モード）。\n"
        "Google Colab の場合は「ランタイム」→「ランタイムのタイプを変更」→「GPU（T4 など）」を選択して再実行してください。"
    )

In [ ]:
!pip install -q "transformers[ja]" datasets seqeval accelerate

---

## 1. 日本語 NER データセットの構築（BIO 形式）

外部の認証付きデータセットをダウンロードしなくても安定して再現できるよう、`PER`（人名）・`ORG`（組織名）・`LOC`（地名）の 3 クラスを含む 114 文の日本語 NER データセットをノートブック内で構築します。
複数エンティティが共起する文・単一エンティティ文・固有表現を含まないネガティブ文をバランスよく含め、学習・検証・テスト（76 / 19 / 19）に層化分割します。

ラベル体系は標準的な **BIO タグ**（7 クラス）を採用します。
- `O`: 固有表現以外
- `B-PER`, `I-PER`: 人名の開始・継続
- `B-ORG`, `I-ORG`: 組織名の開始・継続
- `B-LOC`, `I-LOC`: 地名の開始・継続

In [ ]:
import random
from collections import Counter

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

LABEL_LIST = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC"]
LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}


def build_ner_dataset():
    """PER / ORG / LOC をバランスよく含む 114 文の日本語 BIO データセットを生成する。"""
    persons = [
        (["山田", "太郎"], "PER"),
        (["佐藤", "花子"], "PER"),
        (["鈴木", "一郎"], "PER"),
        (["高橋", "美咲"], "PER"),
        (["田中", "健太"], "PER"),
        (["伊藤", "直樹"], "PER"),
        (["渡辺", "由美"], "PER"),
        (["中村", "翔太"], "PER"),
        (["小林", "愛"], "PER"),
        (["加藤", "大輔"], "PER"),
        (["吉田", "真由"], "PER"),
        (["山本", "拓海"], "PER"),
    ]
    orgs = [
        (["東京", "大学"], "ORG"),
        (["京都", "大学"], "ORG"),
        (["東北", "大学"], "ORG"),
        (["日本", "銀行"], "ORG"),
        (["国立", "情報学", "研究所"], "ORG"),
        (["理化学", "研究所"], "ORG"),
        (["ソニー", "グループ"], "ORG"),
        (["トヨタ", "自動車"], "ORG"),
        (["日本", "電気"], "ORG"),
        (["ソフトバンク"], "ORG"),
        (["朝日", "新聞社"], "ORG"),
        (["経済", "産業省"], "ORG"),
    ]
    locs = [
        (["東京"], "LOC"),
        (["大阪"], "LOC"),
        (["札幌"], "LOC"),
        (["福岡"], "LOC"),
        (["仙台"], "LOC"),
        (["京都"], "LOC"),
        (["横浜"], "LOC"),
        (["名古屋"], "LOC"),
        (["北海道"], "LOC"),
        (["沖縄"], "LOC"),
        (["日本"], "LOC"),
        (["アメリカ"], "LOC"),
    ]

    def tag_entity(ent_tokens, ent_type):
        return [f"B-{ent_type}"] + [f"I-{ent_type}"] * (len(ent_tokens) - 1)

    def make_sample(parts):
        tokens, tags = [], []
        for item in parts:
            if isinstance(item, tuple):
                ent_toks, ent_type = item
                tokens.extend(ent_toks)
                tags.extend(tag_entity(ent_toks, ent_type))
            else:
                tokens.append(item)
                tags.append("O")
        return {"tokens": tokens, "ner_tags": [LABEL2ID[t] for t in tags], "tag_names": tags}

    samples = []

    # パターン 1: PER + ORG + LOC (24 文)
    for i in range(12):
        p, o, l = persons[i], orgs[i], locs[i]
        samples.append(make_sample([p, "さん", "は", o, "の", "研究", "員", "として", l, "で", "講演", "し", "た", "。"]))
        p2, o2, l2 = persons[(i + 3) % 12], orgs[(i + 5) % 12], locs[(i + 7) % 12]
        samples.append(make_sample([l2, "出身", "の", p2, "氏", "が", o2, "の", "新しい", "プロジェクト", "を", "率いる", "。"]))

    # パターン 2: ORG + LOC (24 文)
    for i in range(12):
        o, l = orgs[i], locs[(i + 2) % 12]
        samples.append(make_sample([o, "は", l, "に", "新しい", "研究", "拠点", "を", "開設", "する", "予定", "だ", "。"]))
        o2, l2 = orgs[(i + 4) % 12], locs[(i + 6) % 12]
        samples.append(make_sample([l2, "で", "開か", "れ", "た", "国際", "会議", "に", o2, "が", "参加", "し", "た", "。"]))

    # パターン 3: PER + LOC (18 文)
    for i in range(9):
        p, l = persons[i], locs[(i + 1) % 12]
        samples.append(make_sample([p, "は", "先週", l, "へ", "出張", "し", "て", "調査", "を", "行っ", "た", "。"]))
        p2, l2 = persons[(i + 4) % 12], locs[(i + 8) % 12]
        samples.append(make_sample(["来年", "の", "春", "に", p2, "さん", "は", l2, "へ", "引っ越す", "らしい", "。"]))

    # パターン 4: PER + ORG (18 文)
    for i in range(9):
        p, o = persons[(i + 2) % 12], orgs[(i + 1) % 12]
        samples.append(make_sample([o, "の", p, "教授", "が", "自然", "言語", "処理", "の", "論文", "を", "発表", "し", "た", "。"]))
        p2, o2 = persons[(i + 5) % 12], orgs[(i + 7) % 12]
        samples.append(make_sample([p2, "氏", "は", o2, "と", "共同", "で", "評価", "データ", "を", "公開", "し", "た", "。"]))

    # パターン 5: 単一エンティティ文 (18 文: LOC/ORG/PER 各 6 文)
    for i in range(6):
        samples.append(make_sample(["昨日", "は", locs[i], "の", "天気", "が", "とても", "良かっ", "た", "。"]))
        samples.append(make_sample([orgs[i], "の", "発表", "内容", "が", "大きな", "話題", "に", "なっ", "た", "。"]))
        samples.append(make_sample([persons[i], "さん", "から", "丁寧", "な", "返信", "メール", "が", "届い", "た", "。"]))

    # パターン 6: 固有表現を含まないネガティブ文 (12 文)
    negative_sentences = [
        ["今日", "の", "午後", "は", "雨", "が", "降る", "予報", "です", "。"],
        ["新しい", "モデル", "の", "学習", "が", "無事", "に", "完了", "し", "た", "。"],
        ["自然", "言語", "処理", "で", "は", "評価", "指標", "の", "選定", "が", "重要", "だ", "。"],
        ["この", "辞書", "に", "は", "専門", "用語", "が", "多く", "登録", "さ", "れ", "て", "いる", "。"],
        ["駅", "の", "近く", "で", "温かい", "コーヒー", "を", "飲ん", "だ", "。"],
        ["データ", "の", "前処理", "と", "クリーニング", "に", "時間", "を", "かけ", "た", "。"],
        ["週末", "は", "図書館", "で", "機械", "学習", "の", "教科書", "を", "読ん", "だ", "。"],
        ["実験", "結果", "を", "表", "と", "グラフ", "に", "まとめ", "て", "共有", "する", "。"],
        ["テスト", "データ", "は", "学習", "時", "に", "決して", "見", "て", "は", "いけ", "ない", "。"],
        ["再現性", "を", "確保", "する", "ため", "に", "乱数", "シード", "を", "固定", "し", "た", "。"],
        ["複数", "の", "作業者", "で", "アノテーション", "の", "一致", "率", "を", "確認", "する", "。"],
        ["未知語", "へ", "の", "対応", "と", "して", "サブワード", "分割", "が", "広く", "使わ", "れる", "。"],
    ]
    for toks in negative_sentences:
        samples.append(make_sample(toks))

    return samples


def validate_bio_sequences(samples):
    """すべての文で長さ一致と BIO 遷移の妥当性を検証する。"""
    for idx, ex in enumerate(samples):
        assert len(ex["tokens"]) == len(ex["ner_tags"]) == len(ex["tag_names"]), f"長さ不一致: {idx}"
        prev = "O"
        for tag in ex["tag_names"]:
            if tag.startswith("I-"):
                ent = tag[2:]
                assert prev in (f"B-{ent}", f"I-{ent}"), f"不正な BIO 遷移 ({prev} -> {tag}) at sample {idx}"
            prev = tag


all_samples = build_ner_dataset()
validate_bio_sequences(all_samples)

# エンティティ構成を層化キーにして train / dev / test (76 / 19 / 19) に分割
strata = [
    "".join(sorted({t[2:] for t in ex["tag_names"] if t != "O"})) or "NONE"
    for ex in all_samples
]
train_samples, temp_samples, _, temp_strata = train_test_split(
    all_samples, strata, test_size=38, random_state=SEED, stratify=strata
)
dev_samples, test_samples = train_test_split(
    temp_samples, test_size=19, random_state=SEED, stratify=temp_strata
)

raw_datasets = DatasetDict({
    "train": Dataset.from_list(train_samples),
    "validation": Dataset.from_list(dev_samples),
    "test": Dataset.from_list(test_samples),
})

print({k: len(v) for k, v in raw_datasets.items()})
tag_dist = Counter(t for ex in all_samples for t in ex["tag_names"])
print("全体タグ分布:", dict(tag_dist))
print("サンプル 0:", list(zip(train_samples[0]["tokens"], train_samples[0]["tag_names"])))

---

## 2. サブワード分割と BIO ラベルの整合（`-100` マスキング）

[`ch3_ner_gazetteer_subword.ipynb`](ch3_ner_gazetteer_subword.ipynb) で確認したように、BERT のトークナイザは 1 つの単語を複数のサブワードに分割することがあります。
PyTorch の `CrossEntropyLoss` はデフォルトでラベル `-100` の位置を損失計算から無視する（`ignore_index=-100`）仕様になっているため、以下のルールで整列します。

- `[CLS]` / `[SEP]` などの特殊トークン → **`-100`**
- 単語の先頭サブワード → **元の BIO ラベル ID**
- 同一単語の 2 つ目以降のサブワード → **`-100`**

なお、`tohoku-nlp/bert-base-japanese-v3` の `BertJapaneseTokenizer` は Python 実装（非 Fast トークナイザ）であるため、`word_ids()` メソッドを持ちません。そこで単語単位でサブワード ID を構築しながら `word_ids` と `labels` を同時に生成する実装にすることで、どの日本語トークナイザでも安全に動作するようにしています。

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


def align_labels_with_tokens(tokens_list, ner_tags_list, tokenizer, max_length=128):
    """事前分かち書き済み単語列をサブワード化し、特殊トークンと後続サブワードに -100 を付与する。"""
    all_input_ids = []
    all_attention_mask = []
    all_token_type_ids = []
    all_labels = []
    all_word_ids = []

    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    unk_id = tokenizer.unk_token_id

    for words, tags in zip(tokens_list, ner_tags_list):
        input_ids = [cls_id]
        labels = [-100]
        word_ids = [None]

        for word_idx, (word, label_id) in enumerate(zip(words, tags)):
            sub_ids = tokenizer.encode(word, add_special_tokens=False)
            if not sub_ids:
                sub_ids = [unk_id]

            # [SEP] 分の 1 トークンを残して max_length を超えないように制御
            if len(input_ids) + len(sub_ids) >= max_length:
                break

            input_ids.extend(sub_ids)
            labels.append(label_id)
            labels.extend([-100] * (len(sub_ids) - 1))
            word_ids.extend([word_idx] * len(sub_ids))

        input_ids.append(sep_id)
        labels.append(-100)
        word_ids.append(None)

        all_input_ids.append(input_ids)
        all_attention_mask.append([1] * len(input_ids))
        all_token_type_ids.append([0] * len(input_ids))
        all_labels.append(labels)
        all_word_ids.append(word_ids)

    return {
        "input_ids": all_input_ids,
        "attention_mask": all_attention_mask,
        "token_type_ids": all_token_type_ids,
        "labels": all_labels,
        "word_ids": all_word_ids,
    }


def preprocess_batch(batch):
    aligned = align_labels_with_tokens(batch["tokens"], batch["ner_tags"], tokenizer)
    # Trainer には word_ids を渡さないため削除
    aligned.pop("word_ids")
    return aligned


# サンプル 1 件でトークンと整列後ラベルの対応を確認
sample_ex = raw_datasets["train"][0]
preview = align_labels_with_tokens([sample_ex["tokens"]], [sample_ex["ner_tags"]], tokenizer)
sub_tokens = tokenizer.convert_ids_to_tokens(preview["input_ids"][0])

preview_df = pd.DataFrame({
    "サブワード": sub_tokens,
    "word_id": preview["word_ids"][0],
    "元単語": [sample_ex["tokens"][w] if w is not None else "-" for w in preview["word_ids"][0]],
    "整列後ラベル": [ID2LABEL[l] if l != -100 else "-100 (無視)" for l in preview["labels"][0]],
})
display(preview_df)

tokenized_datasets = raw_datasets.map(
    preprocess_batch,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)
print(tokenized_datasets)

---

## 3. `AutoModelForTokenClassification` の学習（Fine-tuning）

`AutoModelForTokenClassification` は、BERT の各トークン出力ベクトル（768 次元）の上に線形分類ヘッド（768 → 7 クラス）を載せたモデルです。
評価指標には固有表現認識のデファクト標準である **`seqeval`** を使用し、以下を同時に計測します。

- **エンティティ単位 F1（スパンの境界・種別の照合）**：開始位置・終了位置・エンティティ種別のすべてが完全一致して初めて正解とみなす厳密評価
- **トークン単位正解率（`token_accuracy`）**：比較用。`O` タグが大半を占めるため高く出やすい点に注目します

In [ ]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
from transformers import (
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)


def decode_predictions(predictions, label_ids):
    """モデル出力ロジットと -100 付き正解ラベルから、seqeval 用の BIO タグ列リストへ復元する。"""
    pred_ids = np.argmax(predictions, axis=-1)
    true_tags, pred_tags = [], []
    for preds, labels in zip(pred_ids, label_ids):
        sent_true, sent_pred = [], []
        for p, l in zip(preds, labels):
            if l == -100:
                continue
            sent_true.append(ID2LABEL[int(l)])
            sent_pred.append(ID2LABEL[int(p)])
        true_tags.append(sent_true)
        pred_tags.append(sent_pred)
    return true_tags, pred_tags


def compute_metrics(eval_pred):
    predictions, label_ids = eval_pred
    true_tags, pred_tags = decode_predictions(predictions, label_ids)

    flat_true = [t for sent in true_tags for t in sent]
    flat_pred = [p for sent in pred_tags for p in sent]
    token_acc = sum(t == p for t, p in zip(flat_true, flat_pred)) / max(len(flat_true), 1)

    return {
        "precision": precision_score(true_tags, pred_tags),
        "recall": recall_score(true_tags, pred_tags),
        "f1": f1_score(true_tags, pred_tags),
        "token_accuracy": token_acc,
    }


model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./bert_ner_output",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=8,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=torch.cuda.is_available(),
    logging_steps=5,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

---

## 4. テストセットでの評価（エンティティ単位 F1 vs トークン単位正解率）

検証セット（`validation`）で最良のエンティティ F1 を記録したチェックポイントを使い、未見のテストセット（`test`）で最終評価を行います。
あわせて、**「すべて `O` と予測するダミーベースライン」** のスコアと比べることで、なぜ固有表現認識ではトークン単位正解率ではなく **エンティティ単位 F1** を使う必要があるのかを確認します。

In [ ]:
test_output = trainer.predict(tokenized_datasets["test"])
true_tags, pred_tags = decode_predictions(test_output.predictions, test_output.label_ids)

print("=== テストセット：エンティティ単位分類レポート (seqeval) ===")
print(classification_report(true_tags, pred_tags, digits=4))

# 「すべて O」と予測するダミーベースラインとの比較
all_o_preds = [["O"] * len(sent) for sent in true_tags]
flat_true = [t for sent in true_tags for t in sent]
flat_pred = [p for sent in pred_tags for p in sent]
flat_o = ["O"] * len(flat_true)

comparison_df = pd.DataFrame([
    {
        "モデル": "BERT Token Classification",
        "トークン単位正解率": round(sum(t == p for t, p in zip(flat_true, flat_pred)) / len(flat_true), 4),
        "エンティティ単位 Precision": round(precision_score(true_tags, pred_tags), 4),
        "エンティティ単位 Recall": round(recall_score(true_tags, pred_tags), 4),
        "エンティティ単位 F1": round(f1_score(true_tags, pred_tags), 4),
    },
    {
        "モデル": "すべて O と予測するベースライン",
        "トークン単位正解率": round(sum(t == p for t, p in zip(flat_true, flat_o)) / len(flat_true), 4),
        "エンティティ単位 Precision": round(precision_score(true_tags, all_o_preds, zero_division=0), 4),
        "エンティティ単位 Recall": round(recall_score(true_tags, all_o_preds, zero_division=0), 4),
        "エンティティ単位 F1": round(f1_score(true_tags, all_o_preds, zero_division=0), 4),
    },
])
display(comparison_df)

上表のとおり、「すべて `O` と予測するベースライン」でも `O` タグが多いため**トークン単位正解率は 70% 前後**に達してしまいますが、**エンティティ単位 F1 は 0.0000** になります。
これが、固有表現認識においてスパン単位の Precision / Recall / F1（`seqeval`）が不可欠な理由です。

---

## 5. エラー分析と任意のテキストでの推論

テストセットで予測が正解と一致しなかった文（エラー例）を抽出し、どのエンティティで境界ズレ・クラス誤認・見逃しが起きているかを確認します。
さらに、学習済みモデルで任意の分かち書き文から固有表現を抽出するヘルパー関数を動かしてみましょう。

In [ ]:
def extract_entities_from_bio(tokens, tags):
    """BIO タグ列から (スパン文字列, エンティティ種別, 開始位置, 終了位置) のリストを抽出する。"""
    entities = []
    cur_type = None
    cur_tokens = []
    start = None

    for i, (tok, tag) in enumerate(zip(tokens, tags)):
        if tag.startswith("B-"):
            if cur_type is not None:
                entities.append(("".join(cur_tokens), cur_type, start, i))
            cur_type = tag[2:]
            cur_tokens = [tok]
            start = i
        elif tag.startswith("I-") and cur_type == tag[2:]:
            cur_tokens.append(tok)
        else:
            if cur_type is not None:
                entities.append(("".join(cur_tokens), cur_type, start, i))
                cur_type = None
                cur_tokens = []
                start = None
    if cur_type is not None:
        entities.append(("".join(cur_tokens), cur_type, start, len(tokens)))
    return entities


# テストセットのエラー分析
error_rows = []
for ex, gold_seq, pred_seq in zip(raw_datasets["test"], true_tags, pred_tags):
    gold_ents = extract_entities_from_bio(ex["tokens"], gold_seq)
    pred_ents = extract_entities_from_bio(ex["tokens"], pred_seq)
    if gold_ents != pred_ents:
        error_rows.append({
            "文": "".join(ex["tokens"]),
            "正解エンティティ": [(s, t) for s, t, _, _ in gold_ents],
            "予測エンティティ": [(s, t) for s, t, _, _ in pred_ents],
        })

print(f"テスト文数: {len(raw_datasets['test'])} | エラー文数: {len(error_rows)}")
if error_rows:
    display(pd.DataFrame(error_rows))
else:
    print("テストセットの全サンプルでエンティティが完全一致しました！")

In [ ]:
def predict_entities(tokens):
    """分かち書き済み単語列に対して固有表現を推論する。"""
    model.eval()
    device = next(model.parameters()).device
    aligned = align_labels_with_tokens([tokens], [[0] * len(tokens)], tokenizer)

    input_ids = torch.tensor(aligned["input_ids"], device=device)
    attention_mask = torch.tensor(aligned["attention_mask"], device=device)
    token_type_ids = torch.tensor(aligned["token_type_ids"], device=device)

    with torch.no_grad():
        logits = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        ).logits[0]

    pred_ids = logits.argmax(dim=-1).cpu().tolist()
    word_tags = [
        ID2LABEL[pid]
        for pid, lbl in zip(pred_ids, aligned["labels"][0])
        if lbl != -100
    ]
    return word_tags, extract_entities_from_bio(tokens, word_tags)


custom_sentences = [
    ["鈴木", "一郎", "さん", "は", "京都", "大学", "から", "福岡", "へ", "移動", "し", "た", "。"],
    ["国立", "情報学", "研究所", "の", "高橋", "美咲", "氏", "が", "東京", "で", "発表", "する", "。"],
    ["昨日", "は", "雨", "の", "中", "で", "評価", "実験", "を", "続け", "た", "。"],
]

for toks in custom_sentences:
    tags, ents = predict_entities(toks)
    print(f"文: {''.join(toks)}")
    print(f"  抽出結果: {[(s, t) for s, t, _, _ in ents]}")

---

## 6. まとめ

- **サブワードと BIO タグの整合（`-100` マスキング）** を行うことで、BERT のトークン分類ヘッドを使って日本語固有表現認識（NER）を端的に学習できます。
- 固有表現認識の評価では、`O` タグの割合が大きいためトークン単位正解率だけでは実力を測れません。**`seqeval` によるエンティティ単位 F1** を使うことが重要です。
- 実用システムでは、本ノートブックのような BERT NER モデルに加えて、ドメイン固有名詞を補う**ガゼッティア（辞書）**や**アノテーション基準の整備**（第 5 章）を組み合わせることで精度をさらに高められます。